In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/

/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능


In [ ]:
!pip install transformers
!pip install torchinfo
!pip install pytorch-lightning
!pip install torchmetrics
!pip install sentenecepiece
!pip install -i https://test.pypi.org/simple/ bitsandbytes
!pip install peft
!pip install accelerate
!pip install -q -U datasets scipy ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.9/776.9 kB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.2/805.2 kB 31.6 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement sentenecepiece (from versions: none)
ERROR: No matching distribution found for sentenecepiece
Looking in indexes: https://test.pypi.org/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 MB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.7/174.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 32.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 54.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 19.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/

In [ ]:
import numpy as np
import pandas as pd
import os, sys, re, math, random, time, json, pickle, tqdm, gc
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import AdamW, get_linear_schedule_with_warmup
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, BertTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from tqdm import tqdm

class args:
    # amp = True
    gpu = '0'

    label_size = 2
    epochs=10
    batch_size=8
    weight_decay=1e-6
    max_len = 100

    start_lr = 2e-5

    num_workers=0
    seed=2022

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
print(device)
##----------------
def set_seeds(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(seed=args.seed)

path = '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/'

df_train = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_train.csv', encoding = 'utf-8')
df_dev = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_dev.csv', encoding = 'utf-8')
df_test = pd.read_csv('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/df_test.csv', encoding = 'utf-8')

df_train.dropna(inplace = True)
df_dev.dropna(inplace = True)
df_test.dropna(inplace = True)

df_train.reset_index(drop = True, inplace = True)
df_dev.reset_index(drop = True, inplace = True)

cuda


In [ ]:
import torch
from transformers import BitsAndBytesConfig

base_model_id1 = "beomi/llama-2-ko-7b"
base_model_id2 = "EleutherAI/polyglot-ko-12.8b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
class TestDataset(Dataset):

    def __init__(self, dataset, text, tokenizer, max_len):
        self.dataset = dataset
        self.text = text
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = self.dataset.iloc[idx]['input']
        inputs = self.tokenizer(text,
                                return_tensors='pt',
                                truncation=True,
                                max_length=self.max_len,
                                padding='max_length',
                                add_special_tokens=True)

        input_ids = inputs['input_ids'][0]
        attention_mask = inputs['attention_mask'][0]

        return input_ids, attention_mask

    def __len__(self):
        return len(self.dataset)

tokenizer1 = AutoTokenizer.from_pretrained(
    base_model_id1,
    model_max_length=100,
    padding_side="right",
    add_eos_token=True)

tokenizer1.eos_token = tokenizer1.eos_token
tokenizer1.pad_token = tokenizer1.pad_token if tokenizer1.pad_token is not None else tokenizer1.eos_token

test_dataset1 = TestDataset(df_test, 'input', tokenizer1, args.max_len)
test_loader1 = DataLoader(test_dataset1, batch_size=args.batch_size, shuffle=False)

tokenizer2 = AutoTokenizer.from_pretrained(
    base_model_id2,
    model_max_length=100,
    padding_side="right",
    add_eos_token=True)

tokenizer2.eos_token = tokenizer2.eos_token
tokenizer2.pad_token = tokenizer2.pad_token if tokenizer2.pad_token is not None else tokenizer2.eos_token

test_dataset2 = TestDataset(df_test, 'input', tokenizer2, args.max_len)
test_loader2 = DataLoader(test_dataset2, batch_size=args.batch_size, shuffle=False)

model_name3 = "beomi/KcELECTRA-base-v2022"
tokenizer3 = AutoTokenizer.from_pretrained(model_name3)

test_dataset3 = TestDataset(df_test, 'input', tokenizer3, args.max_len)
test_loader3 = DataLoader(test_dataset3, batch_size=args.batch_size, shuffle=False)

model_name4 = "snunlp/KR-Medium"
tokenizer4 = BertTokenizer.from_pretrained(model_name4)

test_dataset4 = TestDataset(df_test, 'input', tokenizer4, args.max_len)
test_loader4 = DataLoader(test_dataset4, batch_size=args.batch_size, shuffle=False)

In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
)
model1 = AutoModelForSequenceClassification.from_pretrained(base_model_id1, quantization_config = bnb_config)
base_model1 = prepare_model_for_kbit_training(model1)
peft_model1 = get_peft_model(base_model1, peft_config)


===================================BUG REPORT===================================
Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes

 and submit this information together with your error trace to: https://github.com/TimDettmers/bitsandbytes/issues
CUDA_SETUP: WARNING! libcudart.so not found in any environmental path. Searching in backup paths...
CUDA SETUP: CUDA runtime path found: /usr/local/cuda/lib64/libcudart.so.11.0
CUDA SETUP: Highest compute capability among GPUs detected: 8.0
CUDA SETUP: Detected CUDA version 118
CUDA SETUP: Loading binary /usr/local/lib/python3.10/dist-packages/bitsandbytes/libbitsandbytes_cuda118.so...


/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: /usr/lib64-nvidia did not contain ['libcudart.so', 'libcudart.so.11.0', 'libcudart.so.12.0'] as expected! Searching further paths...
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('8013'), PosixPath('//172.28.0.1'), PosixPath('http')}
  warn(msg)
/usr/local/lib/python3.10/dist-packages/bitsandbytes/cuda_setup/main.py:147: UserWarning: WARNING: The following directories listed in your path were found to be non-existent: {PosixPath('--logtostderr --listen_host=172.28.0.12 --

Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at beomi/llama-2-ko-7b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, inference_mode=False, r=8, lora_alpha=16, lora_dropout=0.05
)

model2 = AutoModelForSequenceClassification.from_pretrained(base_model_id2, quantization_config = bnb_config)
base_model2 = prepare_model_for_kbit_training(model2)
peft_model2 = get_peft_model(base_model2, peft_config)

Loading checkpoint shards:   0%|          | 0/28 [00:00<?, ?it/s]

Some weights of GPTNeoXForSequenceClassification were not initialized from the model checkpoint at EleutherAI/polyglot-ko-12.8b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model3 = AutoModelForSequenceClassification.from_pretrained(model_name3, num_labels=args.label_size)
model4 = AutoModelForSequenceClassification.from_pretrained(model_name4, num_labels=args.label_size)

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at snunlp/KR-Medium and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
peft_model1.config.pad_token_id = tokenizer1.pad_token_id
peft_model2.config.pad_token_id = tokenizer2.pad_token_id
model3.config.pad_token_id = tokenizer3.pad_token_id
model4.config.pad_token_id = tokenizer3.pad_token_id

model_paths1 = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_llama2_ko_7b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_llama2_ko_7b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_llama2_ko_7b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_llama2_ko_7b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_llama2_ko_7b_kfold.pt',
               ]
model_paths2 = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_polyglot_ko_12.8b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_polyglot_ko_12.8b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_polyglot_ko_12.8b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_polyglot_ko_12.8b_kfold.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_polyglot_ko_12.8b_kfold.pt'
               ]
model_paths3 = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_kc_electra.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_kc_electra.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_kc_electra.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_kc_electra.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_kc_electra.pt'
               ]
model_paths4 = ['/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold1_krbert.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold2_krbert.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold3_krbert.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold4_krbert.pt',
               '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/fold5_krbert.pt'
               ]

models1 = []
models2 = []
models3 = []
models4 = []

# 모델1 로드
for model_path in model_paths1:
    peft_model1.load_state_dict(torch.load(model_path))
    peft_model1.to(device)
    peft_model1.eval()
    models1.append(peft_model1)
    torch.cuda.empty_cache()
    gc.collect()

# 모델2 로드
for model_path in model_paths2:
    peft_model2.load_state_dict(torch.load(model_path))
    peft_model2.to(device)
    peft_model2.eval()
    models2.append(peft_model2)
    torch.cuda.empty_cache()
    gc.collect()

for model_path in model_paths3:
    model3.load_state_dict(torch.load(model_path))
    model3.to(device)
    model3.eval()
    models3.append(model3)
    torch.cuda.empty_cache()
    gc.collect()

for model_path in model_paths4:
    model4.load_state_dict(torch.load(model_path))
    model4.to(device)
    model4.eval()
    models4.append(model4)
    torch.cuda.empty_cache()
    gc.collect()

res = []
with torch.no_grad():
    for i, (data1, data2, data3, data4) in enumerate(tqdm(zip(test_loader1, test_loader2, test_loader3, test_loader4))):
        i += 1
        ids1, mask1 = data1[0].to(device), data1[1].to(device)
        ids2, mask2 = data2[0].to(device), data2[1].to(device)
        ids3, mask3 = data3[0].to(device), data3[1].to(device)
        ids4, mask4 = data4[0].to(device), data4[1].to(device)

        outputs1 = [model(input_ids=ids1, attention_mask=mask1)[0] for model in models1]
        outputs2 = [model(input_ids=ids2, attention_mask=mask2)[0] for model in models2]
        outputs3 = [model(input_ids=ids3, attention_mask=mask3)[0] for model in models3]
        outputs4 = [model(input_ids=ids4, attention_mask=mask4)[0] for model in models4]
        outputs = outputs1 + outputs2 + outputs3 + outputs4

        avg_output = sum(outputs) / len(outputs)
        avg_output = avg_output.cpu().numpy()

        res.extend(avg_output.argmax(axis=1).tolist())

        torch.cuda.empty_cache()
        gc.collect()

df_test['output'] = res

259it [09:24,  2.18s/it]


In [ ]:
df_test

,Unnamed: 0,id,input,output
0,0,nikluge-au-2022-test-000001,극좌는 이 비겁자층을 제대로 요리할 줄 안다...,1
1,1,nikluge-au-2022-test-000002,내가 진짜 올 해 안에 차 산다!,0
2,2,nikluge-au-2022-test-000003,"선거 때마다 불장난 하는 못된 버릇 대대손손 배워가지고 그러고 까불어대면, 너 나중...",1
3,3,nikluge-au-2022-test-000004,"난 99년도에 이미 세상은 망해서 선한자들은 이미 모두 하늘로 올라갔고, 남은 우리...",1
4,4,nikluge-au-2022-test-000005,피의자로 가는 싸가지 없는 쓰래기!,1
...,...,...,...,...
2067,2067,nikluge-au-2022-test-002068,근데 비계 올 사람만 찍어,0
2068,2068,nikluge-au-2022-test-002069,지들 입맛대로 막 가네 미친,1
2069,2069,nikluge-au-2022-test-002070,얘는 지가 이걸 쓰면서 왜 이런 생각을 못하지?,1
2070,2070,nikluge-au-2022-test-002071,알겟나요,0


In [ ]:
def jsonlload(fname):
    with open(fname, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        j_list = [json.loads(line) for line in lines]
    return j_list

test_df = jsonlload('/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/NIKL_AU_2023_COMPETITION_v1.0/nikluge-au-2022-test.jsonl')
len(test_df)

2072

In [ ]:
for i, p in enumerate(df_test['output']):
    test_df[i]['output'] = p

In [ ]:
def jsonldump(j_list, fname):
    with open(fname, "w", encoding='utf-8') as f:
        for json_data in j_list:
            f.write(json.dumps(json_data, ensure_ascii=False)+'\n')
jsonldump(test_df, '/content/drive/MyDrive/연세_수업자료/2학기/텍스트이해와인공지능/기말과제/test24_kfold_krbert.jsonl')